# Change from V2.0:
## Updated Secondary Label detection and allication in labeling stage

In [2]:
# %% [markdown]
# # RQ2 — Step 0: Clone repositories
# Reads a CSV with column `repo_url`, clones/fetches into CLONE_ROOT,
# and writes a manifest with basic metadata.

# %%
from __future__ import annotations
import csv, subprocess, sys, json, time
from pathlib import Path
from typing import Optional, List
from pathlib import Path

# -----------------------------
# Config (edit as needed)
# -----------------------------


# Use a raw string r"..." for Windows paths with spaces
WORK_ROOT    = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
URL_LIST_CSV = WORK_ROOT / "URL_List.csv"   # put your CSV here
CLONE_ROOT   = WORK_ROOT / "clones"         # repos will clone here
MANIFEST_CSV = WORK_ROOT / "clones_manifest.csv"

WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)


# Create dirs
WORK_ROOT.mkdir(parents=True, exist_ok=True)
CLONE_ROOT.mkdir(parents=True, exist_ok=True)

# %%
def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    return subprocess.run(cmd, cwd=cwd, check=check, capture_output=True, text=True)

def repo_dir_name_from_url(url: str) -> str:
    # e.g. https://github.com/owner/name(.git) -> owner__name
    base = url.split("//")[-1]
    parts = base.split("/")
    if len(parts) >= 3:
        owner = parts[-2]
        name  = parts[-1].replace(".git", "")
        return f"{owner}__{name}"
    return base.replace("/", "__").replace(".git", "")

def ensure_cloned(url: str, dest_root: Path) -> Path:
    dest_root.mkdir(parents=True, exist_ok=True)
    d = dest_root / repo_dir_name_from_url(url)
    if d.exists() and (d / ".git").exists():
        # Refresh remote info (best-effort)
        try:
            sh(["git", "fetch", "--all", "--tags", "--prune"], cwd=d)
        except Exception:
            pass
        return d
    sh(["git", "clone", "--no-tags", "--filter=blob:none", "--recurse-submodules=no", url, str(d)])
    return d

def get_total_commits(repo_dir: Path) -> int:
    cp = sh(["git", "rev-list", "--all", "--count"], cwd=repo_dir)
    return int(cp.stdout.strip() or "0")

# %%
assert URL_LIST_CSV.exists(), f"CSV not found: {URL_LIST_CSV}"

rows, ok, fail = [], 0, 0
with URL_LIST_CSV.open(newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        url = (row.get("repo_url") or "").strip()
        if not url:
            continue
        t0 = time.time()
        rec = {"repo_url": url, "dir": None, "status": "unknown", "seconds": None, "total_commits": None, "error": ""}
        try:
            d = ensure_cloned(url, CLONE_ROOT)
            rec["dir"] = str(d)
            rec["total_commits"] = get_total_commits(d)
            rec["status"] = "ok"
            ok += 1
        except subprocess.CalledProcessError as e:
            rec["status"] = "error"
            rec["error"]  = (e.stderr or e.stdout or str(e)).strip()[:2000]
            fail += 1
        rec["seconds"] = round(time.time() - t0, 2)
        rows.append(rec)
        print(f"[{rec['status']}] {url} -> {rec['dir']} ({rec['seconds']}s)")

# %%
# Write manifest
MANIFEST_CSV.parent.mkdir(parents=True, exist_ok=True)
with MANIFEST_CSV.open("w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["repo_url","dir","status","seconds","total_commits","error"])
    w.writeheader()
    w.writerows(rows)

print(f"\nDone. OK={ok}, FAIL={fail}. Manifest: {MANIFEST_CSV}")


[ok] https://github.com/connectbot/connectbot -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\connectbot__connectbot (1.64s)
[ok] https://github.com/robolectric/robolectric -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\robolectric__robolectric (7.22s)
[ok] https://github.com/opendocument-app/OpenDocument.droid -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\opendocument-app__OpenDocument.droid (4.29s)
[ok] https://github.com/maxpower47/PinDroid -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\maxpower47__PinDroid (2.95s)
[ok] https://github.com/Rajawali/Rajawali -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\Rajawali__Rajawali (6.46s)
[ok] https://github.com/cgeo/cgeo -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\cgeo__cgeo (24.03s)
[ok] https://github.com/OneBusAway/onebusaway-android -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones\OneBusAway__onebus

In [25]:
# # RQ2 — Step 1: Mining commits touching CI/Gradle/Scripts

from __future__ import annotations

import os
import re
import json
import subprocess
import datetime as dt
from pathlib import Path
from typing import List, Tuple, Optional, Dict, Any, Iterable
from collections import Counter, defaultdict

# =============================
# Config (edit as needed)
# =============================
WORK_ROOT      = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
CLONE_ROOT     = WORK_ROOT / "clones"
SNAPSHOT_DIR   = WORK_ROOT / "snapshots"
MAX_COMMITS_PER_REPO = 0  # 0 = no limit

# Outputs
WRITE_PER_FILE_AUDIT     = True   # {repo}.jsonl (per-file, audit/validation)
WRITE_COMMIT_SNAPSHOTS   = True   # {repo}.commit.jsonl (per-commit, for CCES)

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

# ---- Commit cutoff (America/Toronto) ----
try:
    from zoneinfo import ZoneInfo  # Python 3.9+
except Exception:
    ZoneInfo = None

CUTOFF_TZ_NAME = "America/Toronto"
_CUTOFF_DATE   = (2025, 8, 10, 23, 59, 59)  # YYYY, M, D, H, M, S local
if ZoneInfo is not None:
    _tz = ZoneInfo(CUTOFF_TZ_NAME)
    _local_dt = dt.datetime(*_CUTOFF_DATE, tzinfo=_tz)
    CUTOFF_EPOCH = int(_local_dt.timestamp())
    CUTOFF_BEFORE_STR = _local_dt.strftime("%Y-%m-%d %H:%M:%S %z")
else:
    _utc_dt = dt.datetime(2025, 8, 11, 3, 59, 59, tzinfo=dt.timezone.utc)  # EDT fallback
    CUTOFF_EPOCH = int(_utc_dt.timestamp())
    CUTOFF_BEFORE_STR = "2025-08-10 23:59:59 -0400"

print(f"[cutoff] Using commit cutoff <= {CUTOFF_BEFORE_STR} (epoch={CUTOFF_EPOCH})")

# Optional YAML
try:
    import yaml  # pip install pyyaml
except Exception:
    yaml = None

# =============================
# Subprocess helper
# =============================
def sh(cmd: List[str], cwd: Optional[Path] = None, check: bool = True) -> subprocess.CompletedProcess:
    env = os.environ.copy()
    env["GIT_PAGER"] = ""
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        check=check,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True,
        encoding="utf-8",
        errors="replace",
        env=env,
    )

# =============================
# Surfaces (CI/Gradle/Scripts)
# =============================
CI_VENDOR_PATTERNS = [
    re.compile(r'(?i)(?:^|/)\.travis\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.appveyor\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)appveyor\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)circle\.yml$'),
    re.compile(r'(?i)(?:^|/)\.circleci/config\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)azure-pipelines\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.github/workflows/.*\.(yml|yaml)$'),
    re.compile(r'(?i)(?:^|/)bitbucket-pipelines\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.gitlab-ci\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)Jenkinsfile(?:\.\w+)?$'),
    re.compile(r'(?i)(?:^|/)bitrise\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)bamboo\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)codeship-services\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.gocd\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)\.cirrus\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)wercker\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)semaphore\.ya?ml$'),
    re.compile(r'(?i)(?:^|/)codemagic\.ya?ml$'),
]
GENERIC_CI_DIRS = re.compile(r'(?i)(?:^|/)(?:ci|\.ci|\.jenkins)(?:/|$)')

def is_ci_file(p: str) -> bool:
    if not p:
        return False
    for rx in CI_VENDOR_PATTERNS:
        if rx.search(p):
            return True
    return bool(GENERIC_CI_DIRS.search(p))

GRADLE_FILES = [
    "build.gradle", "build.gradle.kts",
    "settings.gradle", "settings.gradle.kts",
    "gradle.properties",
    "gradle/wrapper/gradle-wrapper.properties",
]
SCRIPT_EXTS = {".sh", ".bash", ".zsh", ".py", ".bat", ".cmd", ".ps1", ".psm1"}

def is_gradle_file(p: str) -> bool:
    lp = p.lower()
    return any(lp.endswith(x) for x in (f.lower() for f in GRADLE_FILES))

def is_script_file(p: str) -> bool:
    lp = p.lower()
    return any(lp.endswith(ext) for ext in SCRIPT_EXTS)

def touched_relevant(paths: List[str]) -> bool:
    for p in paths:
        if not p.strip():
            continue
        if is_ci_file(p) or is_gradle_file(p) or is_script_file(p):
            return True
    return False

# =============================
# Git helpers
# =============================
def list_relevant_commits(repo_dir: Path) -> List[Tuple[str, int, List[str]]]:
    cp = sh([
        "git", "-c", "i18n.logOutputEncoding=UTF-8", "-c", "core.quotepath=off",
        "log", "--all",
        "--before", CUTOFF_BEFORE_STR,
        "--name-only", "--pretty=%H%x09%ct"
    ], cwd=repo_dir)
    results: List[Tuple[str, int, List[str]]] = []
    sha: Optional[str] = None
    ts: Optional[int] = None
    changed: List[str] = []
    for line in cp.stdout.splitlines():
        if re.match(r"^[0-9a-f]{40}\t\d+$", line):
            if sha is not None and ts is not None and ts <= CUTOFF_EPOCH and touched_relevant(changed):
                results.append((sha, ts, changed))
            sha, ts_s = line.split("\t", 1)
            ts = int(ts_s)
            changed = []
        else:
            if line.strip():
                changed.append(line.strip())
    if sha is not None and ts is not None and ts <= CUTOFF_EPOCH and touched_relevant(changed):
        results.append((sha, ts, changed))
    results.reverse()
    return results

def git_show(repo_dir: Path, sha: str, path: str) -> Optional[str]:
    try:
        cp = sh(["git", "show", f"{sha}:{path}"], cwd=repo_dir)
        return cp.stdout
    except subprocess.CalledProcessError:
        return None

def git_subject(repo_dir: Path, sha: str) -> str:
    try:
        cp = sh(["git", "-c", "i18n.logOutputEncoding=UTF-8", "show", "-s", "--format=%s", sha],
                cwd=repo_dir, check=False)
        return (cp.stdout or "").strip()
    except Exception:
        return ""

# =============================
# Heuristics & extractors
# =============================
RE_INT = re.compile(r"\d+")

YAML_KEYS = {
    "api": ["api-level","apilevel","api_level"],
    "abi": ["abi","arch","cpu","abi_filters","abi-filter"],
    "system_image": ["system-image","target","systemimage"],
    "device": ["device","avd-name","avd","device-profile","model","hardwareProfile"],
    "orchestrator": ["orchestrator","android-test-orchestrator","use-orchestrator"],
    "wait": ["wait-for-boot","wait_for_boot"],
    "timeouts": ["emulator-boot-timeout","timeout","test-timeout","emulator_timeout"],
    "retries": ["retry","retries","max-retries"],
    "matrix": ["matrix","strategy"],
    "runner_os": ["runs-on","machine","image"],
    "jdk": ["java-version","jdk","java","distribution"],
    "invocation": ["run","gradle_args","gradlew_args","task","tasks"],
}

DIY_RX = re.compile(
    r"(?:\bemulator(?:\.bat)?\s-|"
    r"\bavdmanager\b|"
    r"\bsdkmanager\b|"
    r"\bcreate\s+avd\b|"
    r"\badb\s+(?:-s\s+\S+\s+)?wait-for-device\b|"
    r"\badb\s+shell\s+getprop\s+sys\.boot_completed\b|"
    r"\bqemu\b)",
    re.I,
)
GMD_RX = re.compile(r"(?:\bmanaged\s+device\b|\bgradle\s+managed\s+device\b|\bPixel\w*Api\d+\b)", re.I)
GMD_GRADLE_RX = re.compile(r"(?:\bmanagedDevices\s*\{|testOptions\s*\{[^}]*devices)", re.I | re.S)
CONNECTED_RX = re.compile(r"\bconnectedAndroidTest\b", re.I)

# GitHub Actions emulator runner
GHA_EMULATOR_USES_RX = re.compile(r"(?i)^(?:reactivecircus/android-emulator-runner)(?:@.+)?$")

PRIORITY_INVOCATION = ["gmd", "diy", "gradle_connected", "unknown"]

# Gradle comment stripping (fixes earlier flags error)
LINE_COMMENT_RX  = re.compile(r"(^|\s)//.*$", re.M)  # compiled with MULTILINE
BLOCK_COMMENT_RX = re.compile(r"/\*.*?\*/", re.S)
def strip_gradle_comments(text: str) -> str:
    t = BLOCK_COMMENT_RX.sub("", text)
    t = LINE_COMMENT_RX.sub("", t)  # do NOT pass flags here (pattern is precompiled)
    return t

AGP_PLUGIN_DSL_RX = re.compile(
    r"""id\s*\(?\s*
        [\"']com\.android\.(?:application|library|test|dynamic-feature)[\"']\s*\)?
        \s*version\s*
        [\"']([^\"']+)[\"']
    """,
    re.I | re.X,
)

ORCHESTRATOR_COORD_RX = re.compile(r"androidx\.test:orchestrator(?::[^\s'\"\)]+)?", re.I)
ORCHESTRATOR_EXEC_RX  = re.compile(r"testOptions\s*\{[^}]*execution\s*['\"]ANDROIDX_TEST_ORCHESTRATOR['\"]", re.I | re.S)
ORCHESTRATOR_FLAG_RX  = re.compile(r"\buseOrchestrator\s*(?:=|\s)\s*true\b", re.I)
ORCHESTRATOR_PROP_RX  = re.compile(r"\bandroid(?:\.testInstrumentationRunnerArguments)?\.use(?:Test)?Orchestrator\s*=\s*true", re.I)

def detect_orchestrator_from_gradle(text: str) -> bool:
    t = strip_gradle_comments(text or "")
    return bool(
        ORCHESTRATOR_COORD_RX.search(t) or
        ORCHESTRATOR_EXEC_RX.search(t)  or
        ORCHESTRATOR_FLAG_RX.search(t)  or
        ORCHESTRATOR_PROP_RX.search(t)
    )

# =============================
# YAML extractor (scoped/unscoped + provenance/purpose)
# =============================
def classify_invocation_style(yaml_snippets: List[str], gradle_texts: List[str]) -> str:
    hay_yaml = "\n".join(s for s in yaml_snippets if s)
    hay_gradle = "\n".join(gradle_texts or [])
    gmd_hit = bool(GMD_RX.search(hay_yaml)) or any(GMD_GRADLE_RX.search(t or "") for t in gradle_texts or [])
    diy_hit = bool(DIY_RX.search(hay_yaml) or DIY_RX.search(hay_gradle))
    connected_hit = bool(CONNECTED_RX.search(hay_yaml) or CONNECTED_RX.search(hay_gradle))
    if gmd_hit: return "gmd"
    if diy_hit: return "diy"
    if connected_hit: return "gradle_connected"
    return "unknown"

def extract_from_yaml_text(text: str, gradle_texts: Optional[List[str]] = None) -> Dict[str, Any]:
    if yaml is None:
        return {}
    try:
        docs = list(yaml.safe_load_all(text))
    except Exception:
        docs = []

    out = {
        # coverage
        "api_levels": set(), "abis": set(), "system_images": set(), "device_profiles": set(),
        # runtime split
        "scoped_runtime": {"wait_for_boot": set(), "retries": set(), "timeouts": {}},
        "unscoped_runtime": {"wait_for_boot": set(), "retries": set(), "timeouts": {}},
        # infra/context
        "orchestrator": None, "runner_os": None, "jdk": None, "matrix_axes": set(),
        # study-defined
        "invocation_style": "unknown",
        # provenance/purpose
        "provenance": { "api_levels": set(), "abis": set(), "system_images": set(), "device_profiles": set(),
                        "timeouts": set(), "retries": set(), "wait_for_boot": set(),
                        "runner_os": set(), "jdk": set(), "orchestrator": set() },
        "purpose": {},
    }
    _invocation_snippets: List[str] = []

    def mark_timeout(bucket: Dict[str, set], key: str, val: Any):
        try:
            k = str(key); v = str(val)
        except Exception:
            return
        bucket.setdefault(k, set()).add(v)

    def is_gha_emulator_step(d: dict) -> bool:
        uses = d.get("uses")
        return isinstance(uses, str) and GHA_EMULATOR_USES_RX.match(uses.strip())

    def is_diy_emulator_run(d: dict) -> bool:
        run = d.get("run")
        if isinstance(run, str) and DIY_RX.search(run):
            return True
        for k in ("script", "commands", "command"):
            v = d.get(k)
            if isinstance(v, str) and DIY_RX.search(v):
                return True
            if isinstance(v, list) and any(isinstance(x, str) and DIY_RX.search(x) for x in v):
                return True
        return False

    def add_coverage(field: str, value: Any, in_emulator_block: bool):
        if field == "api_levels":
            out["api_levels"].add(int(value))
        elif field == "abis":
            out["abis"].add(str(value).strip())
        elif field == "system_images":
            out["system_images"].add(str(value).strip())
        elif field == "device_profiles":
            out["device_profiles"].add(str(value).strip())
        out["provenance"][field].add("yaml_scoped" if in_emulator_block else "yaml_unscoped")
        out["purpose"][field] = "emulator_intent"

    def scan_obj(obj: Any, in_emulator_block: bool = False):
        if isinstance(obj, dict):
            emulator_here = is_gha_emulator_step(obj) or is_diy_emulator_run(obj)
            inside = in_emulator_block or emulator_here

            for k, v in obj.items():
                lk = str(k).lower()

                # coverage anywhere (with provenance)
                if lk in (x.lower() for x in YAML_KEYS["api"]):
                    if isinstance(v, list):
                        for x in v:
                            m = RE_INT.search(str(x))
                            if m: add_coverage("api_levels", int(m.group()), inside)
                    else:
                        m = RE_INT.search(str(v))
                        if m: add_coverage("api_levels", int(m.group()), inside)

                if lk in (x.lower() for x in YAML_KEYS["abi"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str): add_coverage("abis", x.strip(), inside)

                if lk in (x.lower() for x in YAML_KEYS["system_image"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str): add_coverage("system_images", x.strip(), inside)

                if lk in (x.lower() for x in YAML_KEYS["device"]):
                    vals = v if isinstance(v, list) else [v]
                    for x in vals:
                        if isinstance(x, str): add_coverage("device_profiles", x.strip(), inside)

                # orchestrator
                if lk in (x.lower() for x in YAML_KEYS["orchestrator"]):
                    if isinstance(v, bool):
                        out["orchestrator"] = v
                    elif isinstance(v, str):
                        out["orchestrator"] = v.lower() in ("1","true","yes","on")
                    out["provenance"]["orchestrator"].add("yaml_scoped" if inside else "yaml_unscoped")
                    out["purpose"]["orchestrator"] = "emulator_intent"

                if lk in (x.lower() for x in YAML_KEYS["matrix"]):
                    if isinstance(v, dict):
                        for ax, _vals in v.items():
                            out["matrix_axes"].add(str(ax))

                if lk in (x.lower() for x in YAML_KEYS["runner_os"]):
                    out["runner_os"] = str(v)
                    out["provenance"]["runner_os"].add("yaml_unscoped")
                    out["purpose"]["runner_os"] = "context_only"

                if lk in (x.lower() for x in YAML_KEYS["jdk"]):
                    out["jdk"] = str(v)
                    out["provenance"]["jdk"].add("yaml_unscoped")
                    out["purpose"]["jdk"] = "context_only"

                if lk in (x.lower() for x in YAML_KEYS["invocation"]):
                    _invocation_snippets.append(str(v))

                # runtime knobs (scoped vs unscoped)
                if lk in (x.lower() for x in YAML_KEYS["wait"]):
                    target = out["scoped_runtime"] if inside else out["unscoped_runtime"]
                    if isinstance(v, bool):
                        target["wait_for_boot"].add(v)
                    elif isinstance(v, str):
                        target["wait_for_boot"].add(v.lower() in ("1","true","yes","on"))
                    out["provenance"]["wait_for_boot"].add("yaml_scoped" if inside else "yaml_unscoped")
                    out["purpose"]["wait_for_boot"] = "emulator_intent" if inside else "context_only"

                if lk in (x.lower() for x in YAML_KEYS["retries"]):
                    try:
                        val = int(RE_INT.search(str(v)).group())
                        target = out["scoped_runtime"] if inside else out["unscoped_runtime"]
                        target["retries"].add(val)
                        out["provenance"]["retries"].add("yaml_scoped" if inside else "yaml_unscoped")
                        out["purpose"]["retries"] = "emulator_intent" if inside else "context_only"
                    except Exception:
                        pass

                if lk in (x.lower() for x in YAML_KEYS["timeouts"]):
                    target = out["scoped_runtime"] if inside else out["unscoped_runtime"]
                    if isinstance(v, dict):
                        for tk, tv in v.items(): mark_timeout(target["timeouts"], tk, tv)
                    else:
                        mark_timeout(target["timeouts"], str(k), v)
                    out["provenance"]["timeouts"].add("yaml_scoped" if inside else "yaml_unscoped")
                    out["purpose"]["timeouts"] = "emulator_intent" if inside else "context_only"

                # recurse
                if isinstance(v, (dict, list)):
                    scan_obj(v, inside)

        elif isinstance(obj, list):
            for x in obj:
                scan_obj(x, in_emulator_block)

    for d in docs:
        scan_obj(d, in_emulator_block=False)

    # classify invocation using YAML snippets + Gradle context (if provided)
    out["invocation_style"] = classify_invocation_style(_invocation_snippets, gradle_texts or [])

    # normalize
    out["api_levels"]      = sorted(out["api_levels"])
    out["abis"]            = sorted(out["abis"])
    out["system_images"]   = sorted(out["system_images"])
    out["device_profiles"] = sorted(out["device_profiles"])
    out["matrix_axes"]     = sorted(out["matrix_axes"])

    for bucket in ("scoped_runtime", "unscoped_runtime"):
        out[bucket]["wait_for_boot"] = sorted(list(out[bucket]["wait_for_boot"]))
        out[bucket]["retries"]       = sorted(list(out[bucket]["retries"]))
        out[bucket]["timeouts"]      = {k: sorted(list(v)) for k, v in out[bucket]["timeouts"].items()}

    out["provenance"] = {k: sorted(list(v)) for k, v in out["provenance"].items()}
    return out

# =============================
# Gradle extractor (+ GMD)
# =============================
def extract_from_gradle_text(path: str, text: str) -> Dict[str, Any]:
    clean = strip_gradle_comments(text or "")
    data: Dict[str, Any] = {"provenance": {}, "purpose": {}}

    # AGP versions (context)
    dep_agp = re.findall(r"com\.android\.tools\.build:gradle:([0-9][^'\"\s\)]+)", clean)
    dsl_agp = AGP_PLUGIN_DSL_RX.findall(clean)
    agp_all = sorted(set(dep_agp + dsl_agp))
    if agp_all:
        data["agp_versions"] = agp_all
        data["purpose"]["agp_versions"] = "context_only"

    # Orchestrator
    if "ANDROIDX_TEST_ORCHESTRATOR" in clean or detect_orchestrator_from_gradle(clean):
        data["orchestrator"] = True
        data.setdefault("provenance", {}).setdefault("orchestrator", []).append("gradle_global")
        data["purpose"]["orchestrator"] = "emulator_intent"

    # GMD DSL present?
    if GMD_GRADLE_RX.search(clean):
        cur = data.get("invocation_style")
        if cur in (None, "", "unknown", "diy", "gradle_connected"):
            data["invocation_style"] = "gmd"

        # coverage via GMD
        for fld in ("api_levels","abis","system_images","device_profiles"):
            data.setdefault(fld, [])
            data.setdefault("provenance", {}).setdefault(fld, [])
            data["purpose"][fld] = "emulator_intent"

        # managedDevices { devices { ... } }
        for md in re.finditer(r"managedDevices\s*\{(.*?)\}", clean, re.I | re.S):
            md_block = md.group(1)
            for devs in re.finditer(r"\bdevices\s*\{(.*?)\}", md_block, re.I | re.S):
                devices_block = devs.group(1)

                # pattern 1: Name(...) { ... }
                for named in re.finditer(r"(?m)^\s*([A-Za-z0-9_]+)\s*\([^)]*\)\s*\{(.*?)\}", devices_block, re.S):
                    dev_name, body = named.group(1), named.group(2)
                    if dev_name not in data["device_profiles"]:
                        data["device_profiles"].append(dev_name)
                    if "gradle_gmd" not in data["provenance"]["device_profiles"]:
                        data["provenance"]["device_profiles"].append("gradle_gmd")

                    for m in re.finditer(r"\bdevice\s*=\s*['\"]([^'\"]+)['\"]", body):
                        dp = m.group(1).strip()
                        if dp and dp not in data["device_profiles"]:
                            data["device_profiles"].append(dp)
                        if "gradle_gmd" not in data["provenance"]["device_profiles"]:
                            data["provenance"]["device_profiles"].append("gradle_gmd")

                    for m in re.finditer(r"\bapiLevel\s*=\s*(\d+)", body, re.I):
                        lvl = int(m.group(1))
                        if lvl not in data["api_levels"]:
                            data["api_levels"].append(lvl)
                        if "gradle_gmd" not in data["provenance"]["api_levels"]:
                            data["provenance"]["api_levels"].append("gradle_gmd")

                    for m in re.finditer(r"\babi\s*=\s*['\"]([^'\"\s]+)['\"]", body, re.I):
                        abi = m.group(1).strip()
                        if abi and abi not in data["abis"]:
                            data["abis"].append(abi)
                        if "gradle_gmd" not in data["provenance"]["abis"]:
                            data["provenance"]["abis"].append("gradle_gmd")

                    for m in re.finditer(r"\bsystemImage(Source|Channel)\s*=\s*['\"]([^'\"\s]+)['\"]", body, re.I):
                        val = m.group(2).strip()
                        if val and val not in data["system_images"]:
                            data["system_images"].append(val)
                        if "gradle_gmd" not in data["provenance"]["system_images"]:
                            data["provenance"]["system_images"].append("gradle_gmd")

                # pattern 2: create("name") { ... }
                for created in re.finditer(r"""create\s*\(\s*['"]([^'"]+)['"]\s*\)\s*\{(.*?)\}""", devices_block, re.I | re.S):
                    dev_name, body = created.group(1), created.group(2)
                    if dev_name not in data["device_profiles"]:
                        data["device_profiles"].append(dev_name)
                    if "gradle_gmd" not in data["provenance"]["device_profiles"]:
                        data["provenance"]["device_profiles"].append("gradle_gmd")

                    for m in re.finditer(r"\bdevice\s*=\s*['\"]([^'\"]+)['\"]", body):
                        dp = m.group(1).strip()
                        if dp and dp not in data["device_profiles"]:
                            data["device_profiles"].append(dp)
                        if "gradle_gmd" not in data["provenance"]["device_profiles"]:
                            data["provenance"]["device_profiles"].append("gradle_gmd")

                    for m in re.finditer(r"\bapiLevel\s*=\s*(\d+)", body, re.I):
                        lvl = int(m.group(1))
                        if lvl not in data["api_levels"]:
                            data["api_levels"].append(lvl)
                        if "gradle_gmd" not in data["provenance"]["api_levels"]:
                            data["provenance"]["api_levels"].append("gradle_gmd")

                    for m in re.finditer(r"\babi\s*=\s*['\"]([^'\"\s]+)['\"]", body, re.I):
                        abi = m.group(1).strip()
                        if abi and abi not in data["abis"]:
                            data["abis"].append(abi)
                        if "gradle_gmd" not in data["provenance"]["abis"]:
                            data["provenance"]["abis"].append("gradle_gmd")

                    for m in re.finditer(r"\bsystemImage(Source|Channel)\s*=\s*['\"]([^'\"\s]+)['\"]", body, re.I):
                        val = m.group(2).strip()
                        if val and val not in data["system_images"]:
                            data["system_images"].append(val)
                        if "gradle_gmd" not in data["provenance"]["system_images"]:
                            data["provenance"]["system_images"].append("gradle_gmd")

    # Gradle wrapper (optional)
    if path.endswith("gradle/wrapper/gradle-wrapper.properties"):
        m = re.search(r"distributionUrl=.*?/gradle-([0-9][\w\.-]+)-", clean)
        if m:
            data["gradle_wrapper_version_raw"] = m.group(1)

    return data

# =============================
# Unified per-file extractor
# =============================
def extract_from_text(path: str, text: str, gradle_context_texts: Optional[List[str]] = None) -> Dict[str, Any]:
    if path.lower().endswith((".yml", ".yaml")) and yaml is not None:
        return extract_from_yaml_text(text, gradle_context_texts or [])
    if path.endswith((
        "build.gradle","build.gradle.kts","gradle.properties",
        "gradle/wrapper/gradle-wrapper.properties","settings.gradle","settings.gradle.kts"
    )):
        return extract_from_gradle_text(path, text)
    return {}

# =============================
# IO helper
# =============================
def write_jsonl(path: Path, rows: Iterable[dict]):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

# =============================
# Coalesce per-commit + evidence
# =============================
def coalesce_commit(rows: List[dict]) -> Dict[str, Any]:
    # Start with base structure
    snap: Dict[str, Any] = {
        "repo": rows[0]["repo"],
        "sha": rows[0]["sha"],
        "timestamp": rows[0]["timestamp"],
        "subject": rows[0].get("subject",""),
        "invocation_style": "unknown",
        "orchestrator": False,
        "api_levels": set(), "abis": set(), "system_images": set(), "device_profiles": set(),
        "runtime": {"wait_for_boot": set(), "retries": set(), "timeouts": {}},  # scoped ONLY
        "runtime_unscoped": {"wait_for_boot": set(), "retries": set(), "timeouts": {}},
        "context": {"runner_os": set(), "jdk": set(), "agp_versions": set()},
        "evidence": [],  # [{path, features_present: [...]}]
    }

    def merge_timeouts(dst: Dict[str, set], src: Dict[str, List[str]]):
        for k, vals in (src or {}).items():
            dst.setdefault(k, set()).update(vals or [])

    for r in rows:
        feats = r.get("features", {}) or {}
        present = []

        # coverage
        for k in ["api_levels","abis","system_images","device_profiles"]:
            vals = feats.get(k)
            if isinstance(vals, list) and vals:
                snap[k].update(vals); present.append(k)

        # orchestrator
        if feats.get("orchestrator") is True:
            snap["orchestrator"] = True
            present.append("orchestrator")

        # invocation (priority)
        ist = feats.get("invocation_style")
        if ist and PRIORITY_INVOCATION.index(ist) < PRIORITY_INVOCATION.index(snap["invocation_style"]):
            snap["invocation_style"] = ist
            present.append(f"invocation:{ist}")

        # context
        if feats.get("runner_os"): snap["context"]["runner_os"].add(str(feats["runner_os"]))
        if feats.get("jdk"):       snap["context"]["jdk"].add(str(feats["jdk"]))
        if isinstance(feats.get("agp_versions"), list):
            if feats["agp_versions"]:
                snap["context"]["agp_versions"].update(feats["agp_versions"])
                present.append("agp_versions")

        # runtime scoped/unscoped
        sr = feats.get("scoped_runtime") or {}
        ur = feats.get("unscoped_runtime") or {}

        if sr.get("wait_for_boot"):
            for v in sr["wait_for_boot"]:
                snap["runtime"]["wait_for_boot"].add(bool(v))
            present.append("wait_for_boot(scoped)")
        if sr.get("retries"):
            for v in sr["retries"]:
                try: snap["runtime"]["retries"].add(int(v))
                except: pass
            present.append("retries(scoped)")
        if sr.get("timeouts"):
            merge_timeouts(snap["runtime"]["timeouts"], sr["timeouts"])
            present.append("timeouts(scoped)")

        if ur.get("wait_for_boot"):
            for v in ur["wait_for_boot"]:
                snap["runtime_unscoped"]["wait_for_boot"].add(bool(v))
            present.append("wait_for_boot(unscoped)")
        if ur.get("retries"):
            for v in ur["retries"]:
                try: snap["runtime_unscoped"]["retries"].add(int(v))
                except: pass
            present.append("retries(unscoped)")
        if ur.get("timeouts"):
            merge_timeouts(snap["runtime_unscoped"]["timeouts"], ur["timeouts"])
            present.append("timeouts(unscoped)")

        if present:
            snap["evidence"].append({"path": r.get("path"), "features_present": sorted(list(set(present)))})

    # normalize sets -> lists
    for k in ["api_levels","abis","system_images","device_profiles"]:
        snap[k] = sorted(snap[k])
    for k in list(snap["context"].keys()):
        snap["context"][k] = sorted(snap["context"][k])
    snap["runtime"]["wait_for_boot"] = sorted(list(snap["runtime"]["wait_for_boot"]))
    snap["runtime"]["retries"]       = sorted(list(snap["runtime"]["retries"]))
    snap["runtime"]["timeouts"]      = {k: sorted(list(v)) for k, v in snap["runtime"]["timeouts"].items()}
    snap["runtime_unscoped"]["wait_for_boot"] = sorted(list(snap["runtime_unscoped"]["wait_for_boot"]))
    snap["runtime_unscoped"]["retries"]       = sorted(list(snap["runtime_unscoped"]["retries"]))
    snap["runtime_unscoped"]["timeouts"]      = {k: sorted(list(v)) for k, v in snap["runtime_unscoped"]["timeouts"].items()}

    # quick confidence flag
    has_emulator_signal = any(len(snap[k]) for k in ["api_levels","abis","system_images","device_profiles"]) or snap["invocation_style"] != "unknown"
    snap["confidence"] = "high" if has_emulator_signal else "low"

    return snap

# =============================
# Main
# =============================
def main():
    repos = [p for p in CLONE_ROOT.iterdir() if (p / ".git").exists()]
    repos.sort(key=lambda p: p.name.lower())
    print(f"Found {len(repos)} repos in {CLONE_ROOT}")

    ok_count = skip_count = err_count = 0

    for repo in repos:
        try:
            rel_commits = list_relevant_commits(repo)
            if MAX_COMMITS_PER_REPO > 0:
                rel_commits = rel_commits[:MAX_COMMITS_PER_REPO]

            per_file_rows: List[dict] = []
            commit_buckets: Dict[str, List[dict]] = defaultdict(list)

            for sha, ts, changed_paths in rel_commits:
                rel_paths = [p for p in changed_paths if is_ci_file(p) or is_gradle_file(p) or is_script_file(p)]
                if not rel_paths:
                    continue

                path_texts: Dict[str, Optional[str]] = {pth: git_show(repo, sha, pth) for pth in rel_paths}
                gradle_texts = [txt for pth, txt in path_texts.items() if txt is not None and is_gradle_file(pth)]
                subj = git_subject(repo, sha)

                for pth in rel_paths:
                    txt = path_texts.get(pth)
                    if txt is None:
                        continue
                    feats = extract_from_text(pth, txt, gradle_context_texts=gradle_texts)
                    row = {
                        "repo": repo.name,
                        "sha": sha,
                        "timestamp": ts,
                        "subject": subj,
                        "path": pth,
                        "features": feats,
                    }
                    per_file_rows.append(row)
                    commit_buckets[sha].append(row)

            if not per_file_rows:
                print(f"[skip] {repo.name}: no relevant snapshots")
                skip_count += 1
                continue

            # Write per-file audit rows
            if WRITE_PER_FILE_AUDIT:
                dst_raw = SNAPSHOT_DIR / f"{repo.name}.jsonl"
                write_jsonl(dst_raw, per_file_rows)
                print(f"[ok] {repo.name}: {len(per_file_rows)} per-file rows -> {dst_raw}")

            # Build + write per-commit snapshots with evidence
            if WRITE_COMMIT_SNAPSHOTS:
                commit_snaps: List[dict] = []
                for (sha, ts, _paths) in rel_commits:
                    rows = commit_buckets.get(sha, [])
                    if not rows:
                        continue
                    commit_snaps.append(coalesce_commit(rows))

                dst_commit = SNAPSHOT_DIR / f"{repo.name}.commit.jsonl"
                write_jsonl(dst_commit, commit_snaps)
                print(f"[ok] {repo.name}: {len(commit_snaps)} commit snapshots -> {dst_commit}")

            ok_count += 1

        except Exception as e:
            print(f"[err] {repo.name}: {e}")
            err_count += 1

    print(f"\nDone. ok={ok_count}, skip={skip_count}, err={err_count}, out_dir={SNAPSHOT_DIR}")

if __name__ == "__main__":
    main()


[cutoff] Using commit cutoff <= 2025-08-10 23:59:59 -0400 (epoch=1754884799)
Found 282 repos in C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\clones
[ok] 4eRTuk__audioview: 78 per-file rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\4eRTuk__audioview.jsonl
[ok] 4eRTuk__audioview: 51 commit snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\4eRTuk__audioview.commit.jsonl
[ok] a-mabe__OpenHIIT: 171 per-file rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\a-mabe__OpenHIIT.jsonl
[ok] a-mabe__OpenHIIT: 134 commit snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\a-mabe__OpenHIIT.commit.jsonl
[ok] a914-gowtham__compose-ratingbar: 159 per-file rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots\a914-gowtham__compose-ratingbar.jsonl
[ok] a914-gowtham__compose-ratingbar: 74 commit snapshots -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\

KeyboardInterrupt: 

In [ ]:
# Step 2 — Labeling (V3-style semantics for all CCEs & fields)
# - Per-path diffing (file-scoped): prevents oscillation from repo-level empties
# - Meaningful-delta gates: require real signal on BOTH sides (coverage/runtime)
# - Versions (AGP/JDK): emit only when magnitude != 0
# - Strict PathFA + arbitration (Delta → PathFA → Subject → Path)

from __future__ import annotations
import json, re
import datetime as _dt
from pathlib import Path
from typing import Dict, Any, List, Tuple, Optional

# -----------------------------
# Config
# -----------------------------
WORK_ROOT    = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
SNAPSHOT_DIR = WORK_ROOT / "snapshots"      # expects per-file audit jsonl(s): {repo}.jsonl
OUT_DIR      = WORK_ROOT / "cce_enriched_V3style"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Helpers
# -----------------------------
VERSION_RE = re.compile(r"\d+(?:\.\d+)*")

def read_perfile_snapshots(folder: Path) -> Dict[str, List[dict]]:
    """
    Load miner per-file audit rows (*.jsonl like {repo}.jsonl), return:
      { repo: [rows sorted by (path, timestamp, sha)] }
    Each row is expected to be of the form the miner wrote:
      {"repo","sha","timestamp","subject","path","features":{...}}
    """
    by_repo: Dict[str, List[dict]] = {}
    for p in folder.glob("*.jsonl"):
        # Skip commit-level rolls (*.commit.jsonl)
        if p.name.endswith(".commit.jsonl"):
            continue
        with p.open(encoding="utf-8") as f:
            for line in f:
                s = line.strip()
                if not s:
                    continue
                d = json.loads(s)
                repo = d.get("repo")
                if not repo:
                    # fallback: derive repo name from filename
                    repo = p.stem
                by_repo.setdefault(repo, []).append(d)
    # sort for deterministic per-path chronological diffs
    for repo, rows in by_repo.items():
        rows.sort(key=lambda r: (r.get("path",""), int(r.get("timestamp",0)), r.get("sha","")))
    return by_repo

def as_set_str(xs) -> set:
    if xs is None: return set()
    if isinstance(xs, (list, set, tuple)):
        return set(str(x) for x in xs if str(x).strip() != "")
    if str(xs).strip() == "": return set()
    return {str(xs)}

def as_set_int(xs) -> set:
    if xs is None: return set()
    out = set()
    seq = xs if isinstance(xs, (list, set, tuple)) else [xs]
    for x in seq:
        try:
            v = int(x)
            out.add(v)
        except:
            pass
    return out

def stringify(x: Any) -> str:
    if isinstance(x, (dict, list, set, tuple)):
        try:
            return json.dumps(x, ensure_ascii=False, sort_keys=True)
        except:
            return str(x)
    return "" if x is None else str(x)

def _safe_json_loads(s: str):
    try:
        return json.loads(s) if s else None
    except:
        return None

def parse_version_tuple(s: str) -> Tuple[int, ...]:
    m = VERSION_RE.search(str(s) if s is not None else "")
    if not m: return tuple()
    parts = m.group(0).split(".")
    out: List[int] = []
    for p in parts:
        try: out.append(int(p))
        except: out.append(0)
    return tuple(out)

def max_version_tuple(strings: List[str]) -> Tuple[int, ...]:
    best: Tuple[int, ...] = tuple()
    for s in strings or []:
        vt = parse_version_tuple(str(s))
        if vt > best: best = vt
    return best

def ensure_utc_epoch(ts_any) -> int:
    try: ts = int(float(ts_any))
    except: ts = 0
    return ts

def epoch_to_iso_utc(ts: int) -> str:
    return _dt.datetime.fromtimestamp(int(ts), tz=_dt.timezone.utc).isoformat().replace("+00:00","Z")

# -----------------------------
# WHAT / Category mapping
# -----------------------------
FIELD_TO_PRIMARY = {
    "api_levels":        "api_bump",
    "agp_versions":      "agp_bump",
    "jdk":               "jdk_bump",
    "runner_os":         "runner_os_change",
    "matrix_axes":       "matrix_change",
    "orchestrator":      "orchestrator_change",
    "timeouts":          "timeout_tuning",
    "retries":           "retry_tuning",
    "device_profiles":   "device_profile_change",
    "abis":              "abi_change",
    "system_images":     "system_image_change",
    "invocation_style":  "invocation_change",
    "wait_for_boot":     "wait_strategy_change",
}

CAT_RUNTIME  = "runtime"
CAT_COVERAGE = "coverage"
CAT_CI       = "ci_platform"
CAT_TOOL     = "toolchain"

CATEGORY_NAME = {
    CAT_RUNTIME:  "Runtime Resilience & Throughput Tuning",
    CAT_COVERAGE: "Test Surface & Coverage Configuration",
    CAT_CI:       "CI Platform & Integrations",
    CAT_TOOL:     "Toolchain & Invocation Evolution",
}

PRIMARY_TO_CATEGORY = {
    "timeout_tuning":         CAT_RUNTIME,
    "retry_tuning":           CAT_RUNTIME,
    "orchestrator_change":    CAT_RUNTIME,
    "wait_strategy_change":   CAT_RUNTIME,

    "api_bump":               CAT_COVERAGE,
    "abi_change":             CAT_COVERAGE,
    "device_profile_change":  CAT_COVERAGE,
    "system_image_change":    CAT_COVERAGE,
    "matrix_change":          CAT_COVERAGE,

    "runner_os_change":       CAT_CI,

    "invocation_change":      CAT_TOOL,
    "agp_bump":               CAT_TOOL,
    "jdk_bump":               CAT_TOOL,

    "other_change":           CAT_CI,  # conservative default
}

# -----------------------------
# Diff engine → per-field CCEs
# (V3-style: per-path; meaningful-only)
# -----------------------------
def diff_features(old: Dict[str, Any], new: Dict[str, Any]) -> List[Dict[str, Any]]:
    old = old or {}; new = new or {}
    out: List[Dict[str, Any]] = []

    def handle_set(field: str, to_set_fn):
        a = to_set_fn(old.get(field)); b = to_set_fn(new.get(field))
        # V3-style: if either side is empty -> ignore as no-signal
        if not a or not b:
            return
        if a == b: return
        added   = sorted(b - a)
        removed = sorted(a - b)
        row = {
            "field": field,
            "old_value": stringify(sorted(a)),
            "new_value": stringify(sorted(b)),
            "change_type": "modified",
        }
        if added:   row["added_items"] = stringify(added)
        if removed: row["removed_items"] = stringify(removed)
        if field == "api_levels":
            try: row["magnitude"] = (max(b) - max(a))
            except: pass
        out.append(row)

    # Coverage & sets
    handle_set("api_levels", as_set_int)
    handle_set("abis", as_set_str)
    handle_set("system_images", as_set_str)
    handle_set("device_profiles", as_set_str)
    handle_set("matrix_axes", as_set_str)  # kept, but will NOT emit coverage intents unless axes are emulator-related

    # AGP versions (ordered string set) — require both sides populated and magnitude != 0
    a_agp = sorted(as_set_str(old.get("agp_versions"))); b_agp = sorted(as_set_str(new.get("agp_versions")))
    if a_agp and b_agp and a_agp != b_agp:
        row = {"field":"agp_versions","old_value":stringify(a_agp),"new_value":stringify(b_agp),"change_type":"modified"}
        old_max = max_version_tuple(list(a_agp)); new_max = max_version_tuple(list(b_agp))
        if old_max or new_max:
            mag = 1 if new_max > old_max else (-1 if new_max < old_max else 0)
            if mag != 0:
                row["magnitude"] = mag
                out.append(row)

    # JDK (scalar/string) — require both sides non-empty and magnitude != 0
    a_jdk_raw = stringify(old.get("jdk"))
    b_jdk_raw = stringify(new.get("jdk"))
    if a_jdk_raw and b_jdk_raw and a_jdk_raw != b_jdk_raw:
        row = {
            "field": "jdk",
            "old_value": a_jdk_raw,
            "new_value": b_jdk_raw,
            "change_type": "modified",
        }
        old_vt = parse_version_tuple(a_jdk_raw)
        new_vt = parse_version_tuple(b_jdk_raw)
        if old_vt or new_vt:
            mag = 1 if new_vt > old_vt else (-1 if new_vt < old_vt else 0)
            if mag != 0:
                row["magnitude"] = mag
                out.append(row)

    # Scalars — require both sides non-None
    for field in ("orchestrator","wait_for_boot","retries","runner_os","invocation_style"):
        a = old.get(field, None); b = new.get(field, None)
        if a is None or b is None:
            continue
        if a != b:
            out.append({
                "field": field, "old_value": stringify(a), "new_value": stringify(b),
                "change_type": "modified",
            })

    # Dict-ish — require both sides non-empty and different
    for field in ("timeouts",):
        a = old.get(field, None); b = new.get(field, None)
        if a in (None, {}) or b in (None, {}):
            continue
        if stringify(a) != stringify(b):
            out.append({
                "field": field, "old_value": stringify(a), "new_value": stringify(b),
                "change_type": "modified",
            })

    return out

# -----------------------------
# Delta whitelist (ONLY these emit Delta intents)
# -----------------------------
DELTA_MEANINGFUL_FIELDS = {
    # coverage
    "api_levels", "abis", "device_profiles", "system_images",
    # runtime knobs
    "timeouts", "retries",
    # invocation flips
    "invocation_style",
    # versions
    "agp_versions", "jdk",
}

# -----------------------------
# Delta-side intents (V3-style gates)
# -----------------------------
_DURATION_RX = re.compile(r"(?i)^\s*(\d+(?:\.\d+)?)\s*(ms|s|m|h)?\s*$")
_UNIT_TO_SEC = {"ms": 0.001, "s": 1, "m": 60, "h": 3600}

def _to_seconds(x) -> Optional[float]:
    if x is None: return None
    if isinstance(x, (int, float)): return float(x)
    m = _DURATION_RX.match(str(x))
    if not m: return None
    val = float(m.group(1)); unit = (m.group(2) or "s").lower()
    return val * _UNIT_TO_SEC.get(unit, 1)

def _sum_timeout_seconds(obj) -> Optional[float]:
    if obj is None: return None
    total = 0.0; seen = 0
    it = obj.values() if isinstance(obj, dict) else (obj if isinstance(obj, list) else [])
    for v in it:
        secs = _to_seconds(v)
        if secs is not None:
            total += secs; seen += 1
    return total if seen else None

def derive_delta_intents(deltas: List[Dict[str, Any]]) -> List[str]:
    labs: set[str] = set()
    for d in deltas:
        field = d.get("field")
        if field not in DELTA_MEANINGFUL_FIELDS:
            continue

        old_v = d.get("old_value"); new_v = d.get("new_value")
        added = _safe_json_loads(d.get("added_items") or "") or []
        removed = _safe_json_loads(d.get("removed_items") or "") or []

        # Coverage sets — both sides populated (already enforced in diff_features)
        if field in {"api_levels","abis","device_profiles","system_images"}:
            if added:   labs.add("expand_coverage")
            if removed: labs.add("reduce_coverage")

        # Timeouts — numeric sum compare
        if field == "timeouts":
            old_obj = _safe_json_loads(old_v); new_obj = _safe_json_loads(new_v)
            old_s = _sum_timeout_seconds(old_obj); new_s = _sum_timeout_seconds(new_obj)
            if old_s is not None and new_s is not None:
                if new_s > old_s: labs.add("flake_mitigation")
                elif new_s < old_s: labs.add("speed_up_ci")

        # Retries — integer compare (already both sides present)
        if field == "retries":
            try:
                a = int(old_v) if isinstance(old_v, str) and old_v.isdigit() else int(_safe_json_loads(old_v))
            except: a = None
            try:
                b = int(new_v) if isinstance(new_v, str) and new_v.isdigit() else int(_safe_json_loads(new_v))
            except: b = None
            if a is not None and b is not None:
                if b > a: labs.add("flake_mitigation")
                elif b < a: labs.add("speed_up_ci")

        # Invocation
        if field == "invocation_style":
            o = (old_v or "").strip().lower()
            n = (new_v or "").strip().lower()
            if n == "gmd" and o != "gmd": labs.add("adopt_gmd")
            if o == "gmd" and n != "gmd": labs.add("drop_gmd")
            if n == "diy" and o != "diy": labs.add("adopt_diy")

        # Versions (AGP/JDK) — only if magnitude present and != 0 (already enforced)
        if field in {"agp_versions","jdk"} and d.get("magnitude") not in (None, 0):
            labs.add("version_change")

    return sorted(labs)

# -----------------------------
# Strict, field-aware PathFA + Subject
# -----------------------------
CI_VENDOR_PATTERNS = [
    (re.compile(r'(?i)(?:^|/)\.travis\.ya?ml$'),               'Travis_CI'),
    (re.compile(r'(?i)(?:^|/)\.appveyor\.ya?ml$'),             'AppVeyor'),
    (re.compile(r'(?i)(?:^|/)/?appveyor\.ya?ml$'),             'AppVeyor'),
    (re.compile(r'(?i)(?:^|/)/?circle\.yml$'),                 'Circle_CI'),
    (re.compile(r'(?i)(?:^|/)\.circleci/config\.ya?ml$'),      'Circle_CI'),
    (re.compile(r'(?i)(?:^|/)/?azure-pipelines\.ya?ml$'),      'Azure_Pipelines'),
    (re.compile(r'(?i)(?:^|/)\.github/workflows/.*\.(yml|yaml)$'), 'GitHub_Actions'),
    (re.compile(r'(?i)(?:^|/)/?bitbucket-pipelines\.ya?ml$'),  'Bitbucket'),
    (re.compile(r'(?i)(?:^|/)\.gitlab-ci\.ya?ml$'),            'GitLab'),
    (re.compile(r'(?i)(?:^|/)/?Jenkinsfile(?:\.ya?ml)?$'),     'Jenkins'),
    (re.compile(r'(?i)(?:^|/)/?bitrise\.ya?ml$'),              'Bitrise'),
    (re.compile(r'(?i)(?:^|/)/?bamboo\.ya?ml$'),               'Bamboo'),
    (re.compile(r'(?i)(?:^|/)/?codeship-services\.ya?ml$'),    'Codeship'),
    (re.compile(r'(?i)(?:^|/)\.gocd\.ya?ml$'),                 'GoCD'),
    (re.compile(r'(?i)(?:^|/)\.cirrus\.ya?ml$'),               'Cirrus'),
    (re.compile(r'(?i)(?:^|/)/?wercker\.ya?ml$'),              'Wercker'),
    (re.compile(r'(?i)(?:^|/)\.semaphore\.ya?ml$'),            'Semaphore'),
    (re.compile(r'(?i)(?:^|/)/?codemagic\.ya?ml$'),            'Nevercode'),
]
GENERIC_CI_DIRS   = re.compile(r'(?i)(?:^|/)(?:ci|\.ci)(?:/|$)')
_GRADLE_PATH_RX   = re.compile(r"(?i)(?:^|/)(?:build|settings)\.gradle(?:\.kts)?$|(?:^|/)gradle\.properties$|(?:^|/)gradle/wrapper/gradle-wrapper\.properties$")
_SCRIPT_PATH_RX   = re.compile(r"(?i)\.(?:sh|py|bat|ps1)$|(?:^|/)(?:scripts?|tools?)(?:/|$)")

# For matrix_axes inspection when used with CI paths
_EMULATOR_AXIS_NAMES = {
    "api","api_level","api-level","apilevel","target_api",
    "abi","abis","arch","cpu","abi_filters","abi-filter",
    "system_image","system-image","systemimage","target",
    "device","device-profile","model","hardwareprofile","avd","avd-name",
}
COVERAGE_FIELDS_DIRECT = {"api_levels","abis","device_profiles","system_images"}
CI_ENV_FIELDS = {"runner_os"}

def detect_intent_path_generic(path: str) -> List[str]:
    intents: List[str] = []
    p = path or ""
    if any(rx.search(p) for rx,_ in CI_VENDOR_PATTERNS) or GENERIC_CI_DIRS.search(p):
        intents.append("ci_env_workflow")
    if _GRADLE_PATH_RX.search(p):
        intents.append("version_change")
    if _SCRIPT_PATH_RX.search(p):
        intents.append("ci_env_workflow")
    return sorted(set(intents))

def _contains_emulator_axes(delta_row: Optional[Dict[str, Any]]) -> bool:
    if not delta_row: return False
    added = _safe_json_loads(delta_row.get("added_items") or "") or []
    removed = _safe_json_loads(delta_row.get("removed_items") or "") or []
    axes = {str(x).strip().lower() for x in (added + removed)}
    return any(ax in _EMULATOR_AXIS_NAMES for ax in axes)

def detect_intent_path_field_aware(path: str,
                                   field: str,
                                   subject: str = "",
                                   delta_row: Optional[Dict[str, Any]] = None) -> Tuple[List[str], Optional[str]]:
    p = path or ""; f = (field or "").lower()
    intents: List[str] = []; reasons: List[str] = []

    vendor = None
    for rx,name in CI_VENDOR_PATTERNS:
        if rx.search(p):
            vendor = name; break
    if vendor is None and GENERIC_CI_DIRS.search(p):
        vendor = "Generic_CI"

    is_gradle_path = bool(_GRADLE_PATH_RX.search(p))
    is_script_path = bool(_SCRIPT_PATH_RX.search(p))

    if vendor:
        if f in COVERAGE_FIELDS_DIRECT:
            intents += ["coverage","ci_env_workflow"]; reasons.append(f"ci={vendor}+coverage_field")
        elif f == "matrix_axes" and _contains_emulator_axes(delta_row):
            intents += ["coverage","ci_env_workflow"]; reasons.append(f"ci={vendor}+matrix_axes(emulator)")
        elif f in CI_ENV_FIELDS:
            intents += ["ci_env_workflow"]; reasons.append(f"ci={vendor}+{f}")

    if is_gradle_path and f in {"agp_versions","jdk"}:
        intents.append("version_change"); reasons.append("gradle/tool path")

    if is_script_path and f in CI_ENV_FIELDS:
        intents.append("ci_env_workflow"); reasons.append("script/tool+ci")

    intents = sorted(set(intents))
    reason = "; ".join(reasons) if reasons else None
    return intents, reason

# -----------------------------
# Subject intents (simple cues)
# -----------------------------
def detect_subject_intents(subject: str) -> Tuple[List[str], List[str], Optional[str]]:
    s = (subject or "").lower()
    hits = set()
    if any(w in s for w in ["flake","flaky","retry","retries","timeout","timeouts","stabil","crash","fix"]):
        hits.add("flake_mitigation")
    if any(w in s for w in ["speed up","faster","reduce time","time to green","parallel","shard","concurr"]) and \
       any(c in s for c in [" ci", "build", "pipeline", "workflow", "runner", "github actions","gitlab","jenkins","circleci","azure pipelines","bitrise"]):
        hits.add("speed_up_ci")
    if any(w in s for w in ["gradle","agp","jdk","java","version","wrapper","target api"]):
        hits.add("version_change")
    if any(w in s for w in ["migrate","switch","replace","port","github actions","gha","gitlab","jenkins","circleci","azure pipelines","bitrise","workflow","pipeline"]):
        hits.add("ci_env_workflow")
    hits = sorted(hits)
    reason = hits[0] if hits else None
    return hits, [], reason

# -----------------------------
# 7-subintent + headline mapping & arbitration
# -----------------------------
RAW_TO_SUBINTENT = {
    "expand_coverage":"coverage", "reduce_coverage":"coverage", "coverage":"coverage",
    "flake_mitigation":"flake_mitigation", "speed_up_ci":"speed_up_ci",
    "orchestrator_change":"orchestrator_change",
    "version_change":"version_change", "adopt_gmd":"invocation_change",
    "drop_gmd":"invocation_change", "adopt_diy":"invocation_change",
    "ci_env_workflow":"ci_platform_infra",
}
SUB_TO_HEADLINE = {
    "coverage":"coverage",
    "flake_mitigation":"runtime",
    "speed_up_ci":"runtime",
    "orchestrator_change":"runtime",
    "version_change":"version_change",
    "invocation_change":"version_change",
    "ci_platform_infra":"ci_platform_infra",
}
SUB_PRIORITY = ["coverage","flake_mitigation","speed_up_ci","version_change","orchestrator_change","invocation_change","ci_platform_infra"]

def to_subbuckets(raws: List[str]) -> List[str]:
    out = []
    for r in raws or []:
        b = RAW_TO_SUBINTENT.get(r)
        if b and b not in out:
            out.append(b)
    return out

def pick_by_priority(cands: List[str]) -> str:
    for b in SUB_PRIORITY:
        if b in cands:
            return b
    return ""

def classify_change_op(old_value: str, new_value: str, change_type: str) -> str:
    ct = (change_type or "").lower()
    if ct == "added":   return "add"
    if ct == "removed": return "remove"
    return "value_edit" if (old_value or "") != (new_value or "") else "no_change"

# -----------------------------
# Main
# -----------------------------
if __name__ == "__main__":
    print(f"[info] per-file snapshots dir: {SNAPSHOT_DIR}")
    print(f"[info] output dir           : {OUT_DIR}")

    by_repo = read_perfile_snapshots(SNAPSHOT_DIR)
    print(f"[info] Loaded per-file snapshots for {len(by_repo)} repos")

    for repo, rows in by_repo.items():
        out_rows: List[dict] = []

        # group by file path (V3 per-path diffs)
        by_path: Dict[str, List[dict]] = {}
        for r in rows:
            by_path.setdefault(r.get("path",""), []).append(r)

        repeat_counter: Dict[Tuple[str,str], int] = {}

        for path, snaps in by_path.items():
            prev: Optional[dict] = None
            for cur in snaps:
                if prev is None:
                    prev = cur
                    continue

                old_feats = prev.get("features", {}) or {}
                new_feats = cur.get("features", {}) or {}

                # V3-style diffs: meaningful-only
                deltas = diff_features(old_feats, new_feats)
                if not deltas:
                    prev = cur
                    continue

                # derived intents from the *whole* transition for aux column
                intent_delta_episode = derive_delta_intents(deltas)

                # per-CCE (per field) labeling
                subject_str = cur.get("subject","")
                ts_epoch_utc = ensure_utc_epoch(cur.get("timestamp", 0))
                ts_iso_utc   = epoch_to_iso_utc(ts_epoch_utc)

                for d in deltas:
                    field = d["field"]

                    # Primary WHAT/category
                    primary_label = FIELD_TO_PRIMARY.get(field, "other_change")
                    primary_category_id = PRIMARY_TO_CATEGORY.get(primary_label, "ci_platform")
                    primary_category = CATEGORY_NAME.get(primary_category_id, primary_category_id)

                    # Per-source (field-gated)
                    intents_delta_list = derive_delta_intents([d]) if field in DELTA_MEANINGFUL_FIELDS else []

                    # Strict PathFA (pass this delta for matrix_axes inspection) + generic path
                    intent_pfa_list, intent_pfa_reason = detect_intent_path_field_aware(path, field, subject_str, d)
                    intent_path_generic = detect_intent_path_generic(path)

                    # Subject
                    intent_subject_all, aux_subject, subj_reason = detect_subject_intents(subject_str)

                    # Normalize
                    intents_delta   = sorted(set(intents_delta_list))
                    intents_pfa     = sorted(set(intent_pfa_list))
                    intents_subject = sorted(set(intent_subject_all))
                    intents_path    = sorted(set(intent_path_generic))

                    # Arbitration: Delta → PathFA → Subject → Path
                    source_to_pool = [
                        ("Delta",   intents_delta,   "delta-derived"),
                        ("PathFA",  intents_pfa,     intent_pfa_reason or "path field-aware"),
                        ("Subject", intents_subject, subj_reason or "subject regex"),
                        ("Path",    intents_path,    "path generic"),
                    ]
                    chosen_source = ""
                    chosen_raws: List[str] = []
                    chosen_reason = ""
                    for s, pool, why in source_to_pool:
                        if pool:
                            chosen_source = s; chosen_raws = pool; chosen_reason = why; break

                    # Driver (7-bucket) + Headline (4-bucket)
                    sub_cands = to_subbuckets(chosen_raws)
                    driver = pick_by_priority(sub_cands) if sub_cands else ""
                    headline = SUB_TO_HEADLINE.get(driver, "") if driver else ""

                    # repeats per (path, field)
                    key = (path, field)
                    repeat_counter[key] = repeat_counter.get(key, 0) + 1
                    repeat_index = repeat_counter[key]
                    repeat_label = "first" if repeat_index == 1 else "repeat"

                    # change_op
                    change_op = classify_change_op(d.get("old_value",""), d.get("new_value",""), d.get("change_type","modified"))

                    out_rows.append({
                        # --- Metadata
                        "repo": repo,
                        "sha": cur.get("sha"),
                        "prev_sha": prev.get("sha"),
                        "timestamp_epoch_utc": ts_epoch_utc,
                        "timestamp_utc": ts_iso_utc,
                        "path": path,
                        "subject": subject_str,

                        # --- Primary (WHAT)
                        "Field": field,
                        "Primary_Label": primary_label,
                        "Primary_Label_Category": primary_category,

                        # --- Diff details (V3-style: no []/None oscillations)
                        "old_value": d.get("old_value",""),
                        "new_value": d.get("new_value",""),
                        "change_type": d.get("change_type","modified"),
                        "change_op": change_op,
                        "magnitude": d.get("magnitude", None),
                        "added_items": d.get("added_items",""),
                        "removed_items": d.get("removed_items",""),
                        "repeat_index": repeat_index,
                        "repeat_label": repeat_label,

                        # --- Per-source intents (strings, comma-joined)
                        "intent_delta": ",".join(intents_delta),
                        "intent_pfa": ",".join(intents_pfa),
                        "intent_subject": ",".join(intents_subject),
                        "intent_path": ",".join(intents_path),
                        "intent_delta_episode": ",".join(sorted(set(intent_delta_episode))) if intent_delta_episode else "",

                        # --- Secondary (chosen source only)
                        "Secondary_Label": ",".join(chosen_raws) if chosen_raws else "",

                        # --- Why (7 + 4)
                        "Driver": driver,
                        "intent_headline": headline,
                        "Driver_Source": chosen_source,
                        "Driver_reason": chosen_reason,

                        # --- Aux (subject aux kept for compat)
                        "Aux_Tags": ",".join(aux_subject) if aux_subject else "",
                        "Driver_Label_Selected": driver,
                    })

                prev = cur

        if not out_rows:
            print(f"[skip] {repo}: no meaningful field-level deltas found")
            continue

        # Write one JSONL per repo (CCE rows)
        dst = OUT_DIR / f"{repo}.jsonl"
        with dst.open("w", encoding="utf-8") as f:
            for r in out_rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        print(f"[ok] {repo}: {len(out_rows)} CCE rows -> {dst}")

    print("Done.")


[info] per-file snapshots dir: C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\snapshots
[info] output dir           : C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V3style
[info] Loaded per-file snapshots for 282 repos
[ok] 4eRTuk__audioview: 8 CCE rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V3style\4eRTuk__audioview.jsonl
[ok] a-mabe__OpenHIIT: 7 CCE rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V3style\a-mabe__OpenHIIT.jsonl
[ok] a914-gowtham__compose-ratingbar: 16 CCE rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V3style\a914-gowtham__compose-ratingbar.jsonl
[ok] AAkira__ExpandableLayout: 5 CCE rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V3style\AAkira__ExpandableLayout.jsonl
[ok] abdelaziz-mahdy__pytorch_lite: 15 CCE rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\cce_enriched_V3style\abdelaziz-mahdy__

In [32]:
# RQ2 — Step 3 (Combine): consolidate per-repo CCE JSONLs (V3-style) into one CSV
from __future__ import annotations

import json, csv
from pathlib import Path
from typing import List, Dict, Any
from datetime import datetime, timezone

# -----------------------------------
# Config
# -----------------------------------
WORK_ROOT   = Path(r"C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2")
# Point this to the directory where Step 2 writes one JSONL per repo (V3-style enriched CCEs)
LABELS_DIR  = WORK_ROOT / "cce_enriched"     # e.g., "cce_enriched" or "cce_enriched_V4.7"
OUT_DIR     = WORK_ROOT / "Combined_CCES"    # output directory for combined CSV
OUT_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV     = OUT_DIR / "dataset_cces.csv"
SCHEMA_VERSION = "v3_style_cces_combine_v1"

# -----------------------------------
# CSV helpers
# -----------------------------------
def _cell(v: Any) -> str:
    """Stringify values for CSV: lists/sets -> comma-joined; dict -> JSON; else str; no [] in cells."""
    if v is None:
        return ""
    if isinstance(v, (list, set, tuple)):
        return ",".join(str(x) for x in v)
    if isinstance(v, dict):
        return json.dumps(v, ensure_ascii=False, sort_keys=True)
    return str(v)

def _write_csv(path: Path, rows: List[dict], preferred_header_order: List[str]):
    # stamp schema_version for consistency
    for r in rows:
        r.setdefault("schema_version", SCHEMA_VERSION)

    if not rows:
        with path.open("w", newline="", encoding="utf-8") as f:
            pass
        return

    # deterministic header order: preferred first, then any extras sorted
    seen = set(preferred_header_order)
    extras = sorted(k for r in rows for k in r.keys() if k not in seen)
    headers = preferred_header_order + extras

    with path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=headers)
        w.writeheader()
        for r in rows:
            w.writerow({k: _cell(r.get(k)) for k in headers})

# -----------------------------------
# Input loader
# -----------------------------------
def read_all_cces(label_dir: Path) -> List[dict]:
    """Read every *.jsonl in LABELS_DIR (one file per repo, V3-style CCE rows)."""
    rows: List[dict] = []
    for p in sorted(label_dir.glob("*.jsonl")):
        with p.open(encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    d = json.loads(line)
                except Exception:
                    continue
                # If repo is missing, infer from filename (basename without extension)
                d.setdefault("repo", p.stem)
                rows.append(d)
    return rows

# -----------------------------------
# Preferred header order (V3-style)
# -----------------------------------
PREFERRED = [
    "schema_version",

    # --- Metadata
    "repo",
    "sha",
    "prev_sha",
    "timestamp_epoch_utc",
    "timestamp_utc",
    "path",
    "subject",

    # --- Primary (WHAT)
    "Field",
    "Primary_Label",
    "Primary_Label_Category",

    # --- Diff details
    "old_value",
    "new_value",
    "change_type",
    "change_op",
    "magnitude",
    "added_items",
    "removed_items",
    "repeat_index",
    "repeat_label",

    # --- Per-source intents (field-level)
    "intent_delta",
    "intent_pfa",
    "intent_subject",
    "intent_path",
    "intent_delta_episode",  # episode-wise delta pool (if your Step 2 includes it)

    # --- Secondary (chosen source only)
    "Secondary_Label",

    # --- Why (driver & headline from chosen source only)
    "Driver",
    "intent_headline",
    "Driver_Source",
    "Driver_reason",

    # --- Aux / traceability
    "Aux_Tags",
    "Intent_Path",            # detailed path intents (if present)
    "Intent_Path_Reason",     # pathFA reasoning (if present)
    "Intent_Subject_Reason",  # subject reasoning (if present)

    # --- Legacy compat (neutralized in V3 approach)
    "Driver_Label_Selected",

    # --- Optional flags if present in your Step 2
    "delta_suppressed",
]

# -----------------------------------
# Main
# -----------------------------------
def main():
    rows = read_all_cces(LABELS_DIR)

    # Stable sort: by timestamp (epoch if present; else parsed ISO; else 0), then repo, sha, Field, path, repeat_index
    def _ts_key(r: dict) -> int:
        t = r.get("timestamp_epoch_utc")
        if t is not None:
            try:
                return int(float(t))
            except Exception:
                pass
        iso = r.get("timestamp_utc")
        if iso:
            try:
                return int(datetime.fromisoformat(iso.replace("Z","+00:00")).replace(tzinfo=timezone.utc).timestamp())
            except Exception:
                pass
        return 0

    rows.sort(key=lambda r: (
        _ts_key(r),
        str(r.get("repo","")),
        str(r.get("sha","")),
        str(r.get("Field","")),
        str(r.get("path","")),
        int(r.get("repeat_index", 0)) if str(r.get("repeat_index","")).isdigit() else 0
    ))

    _write_csv(OUT_CSV, rows, PREFERRED)
    print(f"[ok] Consolidated CCES dataset: {len(rows)} rows -> {OUT_CSV}")

if __name__ == "__main__":
    main()


[ok] Consolidated CCES dataset: 8390 rows -> C:\Android Mobile App\Step2_Clone_Repo\Type_1\Aug_10\RQ2\Combined_CCES\dataset_cces.csv
